FILE NÀY CHỦ YẾU LÀ ĐỂ DỰ ĐOÁN TỪ MODEL, LÚC TRAIN XONG HOẶC CHƯA XONG, DO EM LƯU TRÊN DRIVER MÔI LẦN NÓ EPOCH. file này chắc chỉ cần đổi lại đường dẫn là nó tự lưu trên máy cj nhé. có cái file submission của anh Hiệu đc 92%. nên là khi xuất file xong thì em có đối chiếu với file đó xem lệch nhiều ko, nó ở cái đoạn bến dưới cùng á cj Mêi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/ModelBiLSTM/'
print("✅ Done - Gắn Drive thành công!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Done - Gắn Drive thành công!


In [ ]:
import pickle
import pandas as pd
import numpy as np
import torch

# Đường dẫn
drive_path = '/content/drive/MyDrive/ModelBiLSTM/'

# Load encoders
with open(drive_path + "label_encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

print("✅ Loaded encoders!")
print("Các cột:", list(encoders.keys()))

✅ Loaded encoders!
Các cột: ['attr_1', 'attr_2', 'attr_3', 'attr_4', 'attr_5', 'attr_6']


In [ ]:
# Tiền xử lý dữ liệu
# Tiền xử lý dữ liệu
import numpy as np
import pandas as pd

class ActionSequenceProcessor:

    def __init__(self, max_len=24):
        self.max_len = max_len
        self.PAD_IDX = 0
        self.action2idx = None
        self.idx2action = None

    # =========================
    # 1. Fit: tạo vocabulary
    # =========================
    def fit(self, X, id_column="id"):

        X_only = X.drop(columns=[id_column]) if id_column in X.columns else X

        all_actions = set()
        for col in X_only.columns:
            values = X_only[col].dropna().astype(int).unique()
            all_actions.update(values)

        all_actions = sorted(list(all_actions))

        self.action2idx = {
            action: idx + 1
            for idx, action in enumerate(all_actions)
        }

        self.idx2action = {
            idx: action
            for action, idx in self.action2idx.items()
        }

        print("Vocabulary size (including PAD):", len(self.action2idx) + 1)
        return self

    # =========================
    # 2. Encode 1 row
    # =========================
    def _encode_row(self, row):
        encoded = []
        for val in row:
            if not np.isnan(val):
                encoded.append(self.action2idx[int(val)])
        return encoded

    # =========================
    # 3. Pad / Truncate
    # =========================
    def _pad_truncate(self, seq):
        if len(seq) > self.max_len:
            return seq[-self.max_len:]
        else:
            return seq + [self.PAD_IDX] * (self.max_len - len(seq))

    # =========================
    # 4. Transform
    # =========================
    def transform(self, X, id_column="id"):

        if self.action2idx is None:
            raise ValueError("Bạn phải fit() trước khi transform()")

        X_only = X.drop(columns=[id_column]) if id_column in X.columns else X

        # Encode
        X_encoded = X_only.apply(
            lambda row: self._encode_row(row),
            axis=1
        )

        # Pad
        X_padded = X_encoded.apply(self._pad_truncate)
        X_padded = np.array(X_padded.tolist())

        # Attention mask
        attention_mask = (X_padded != self.PAD_IDX).astype(int)

        # =============================
        # Statistical features
        # =============================

        seq_length = X_encoded.apply(len)

        unique_ratio = X_encoded.apply(
            lambda seq: len(set(seq)) / len(seq) if len(seq) > 0 else 0
        )

        repetition_ratio = 1 - unique_ratio

        first_action = X_encoded.apply(
            lambda seq: seq[0] if len(seq) > 0 else 0
        )

        last_action = X_encoded.apply(
            lambda seq: seq[-1] if len(seq) > 0 else 0
        )

        X_stat_features = pd.DataFrame({
            "seq_length": seq_length,
            "unique_ratio": unique_ratio,
            "repetition_ratio": repetition_ratio,
            "first_action": first_action,
            "last_action": last_action
        })

        return X_padded, attention_mask, X_stat_features

    # =========================
    # 5. Fit + Transform
    # =========================
    def fit_transform(self, X, id_column="id"):
        self.fit(X, id_column)
        return self.transform(X, id_column)

In [ ]:
# =====================
# Bước 3: Đọc và gộp X_train + X_val để fit processor
# =====================
X_train = pd.read_csv('X_train.csv')
X_val = pd.read_csv('X_val.csv')
X_full_train = pd.concat([X_train, X_val])  # Gộp lại như lúc train

print(f"✅ X_full_train shape: {X_full_train.shape}")

# Fit processor trên FULL_TRAIN
processor = ActionSequenceProcessor(max_len=24)
processor.fit(X_full_train)
print(f"✅ Processor fit xong! Vocab size: {len(processor.action2idx) + 1}")

✅ X_full_train shape: (58200, 38)
Vocabulary size (including PAD): 255
✅ Processor fit xong! Vocab size: 255


In [ ]:
X_test = pd.read_csv('X_test.csv')
print(f"✅ Đã đọc X_test: {X_test.shape}")

✅ Đã đọc X_test: (38000, 38)


In [ ]:
X_id = X_test['id']
X_id

,id
0,gpbfd
1,w22ee
2,wyw95
3,izx4w
4,c6o2d
...,...
37995,lfgzf
37996,cs1tz
37997,mw26u
37998,gqnbh


In [ ]:
# Đọc X_test
X_test = pd.read_csv('X_test.csv')
print(f"✅ Đã đọc X_test: {X_test.shape}")

# Transform X_test bằng processor vừa fit
X_seq_test, X_mask_test, X_stat_test = processor.transform(X_test)

print(f"X_seq_test shape: {X_seq_test.shape}")
print(f"X_mask_test shape: {X_mask_test.shape}")
print(f"X_stat_test shape: {X_stat_test.shape}")

✅ Đã đọc X_test: (38000, 38)
X_seq_test shape: (38000, 24)
X_mask_test shape: (38000, 24)
X_stat_test shape: (38000, 5)


Đoạn này thì em đang phải định nghĩa lại phần model, thực chất là em copy từ cái file train kia sang, do trc ko định nghĩa lại thì em thấy nó thông báo model ko define =)))))))))))

In [ ]:
# =====================
# ĐỊNH NGHĨA CLASS GCN
# =====================
class ImprovedGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(dropout)
        self.use_residual = (in_dim == out_dim)

    def forward(self, x, adj):
        h = torch.bmm(adj, x)
        h = self.linear(h)
        if self.use_residual:
            h = h + x
        h = self.norm(h)
        h = F.relu(h)
        h = self.dropout(h)
        return h


# =====================
# ĐỊNH NGHĨA CLASS MODEL (GIỐNG LÚC TRAIN)
# =====================
class ImprovedBiLSTM_GCN_Model(nn.Module):
    def __init__(
        self,
        num_actions,
        num_classes,
        embed_dim=128,
        lstm_hidden=128,
        gcn_hidden=128,
        stat_dim=5,
        dropout=0.15,        # Giống train
        use_attention=False   # Giống train
    ):
        super().__init__()

        if num_classes is None:
            raise ValueError("num_classes must be provided")

        # Embedding
        self.embedding = nn.Embedding(
            num_embeddings=num_actions,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        self.embed_dropout = nn.Dropout(dropout)

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.lstm_dropout = nn.Dropout(dropout)

        # GCN Layers
        self.gcn1 = ImprovedGCNLayer(embed_dim, gcn_hidden, dropout=dropout)
        self.gcn2 = ImprovedGCNLayer(gcn_hidden, gcn_hidden, dropout=dropout)
        self.gcn_dropout = nn.Dropout(dropout)

        # Fusion
        fusion_dim = lstm_hidden * 2 + gcn_hidden + stat_dim
        self.fc_shared = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Multi-head output
        self.heads = nn.ModuleList([
            nn.Linear(128, c) for c in num_classes
        ])

    def build_adj_matrix(self, mask):
        batch_size, seq_len = mask.size()
        device = mask.device

        adj = torch.zeros(batch_size, seq_len, seq_len, device=device)

        for i in range(seq_len - 1):
            adj[:, i, i+1] = 1
            adj[:, i+1, i] = 1

        adj = adj * mask.unsqueeze(1) * mask.unsqueeze(2)
        row_sum = adj.sum(dim=-1, keepdim=True) + 1e-8
        adj = adj / row_sum

        return adj

    def forward(self, x_seq, mask, stat_feat):
        emb = self.embedding(x_seq)
        emb = self.embed_dropout(emb)

        # BiLSTM
        lstm_out, _ = self.lstm(emb)
        lstm_out = lstm_out * mask.unsqueeze(-1)
        lstm_out = self.lstm_dropout(lstm_out)
        lstm_feat = torch.sum(lstm_out, dim=1) / (mask.sum(dim=1, keepdim=True) + 1e-8)

        # GCN
        adj = self.build_adj_matrix(mask)
        gcn_out = self.gcn1(emb, adj)
        gcn_out = self.gcn2(gcn_out, adj)
        gcn_out = gcn_out * mask.unsqueeze(-1)
        gcn_out = self.gcn_dropout(gcn_out)
        gcn_feat = torch.sum(gcn_out, dim=1) / (mask.sum(dim=1, keepdim=True) + 1e-8)

        # Fusion
        fused = torch.cat([lstm_feat, gcn_feat, stat_feat], dim=1)
        shared = self.fc_shared(fused)

        outputs = [head(shared) for head in self.heads]
        return outputs

In [ ]:
# Chuyển sang tensor và đưa lên GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Chuyển sang tensor (bỏ .values)
# Chuyển từ DataFrame sang numpy array trước
X_seq_test = torch.tensor(X_seq_test, dtype=torch.long).to(device)
X_mask_test = torch.tensor(X_mask_test, dtype=torch.float32).to(device)
X_stat_test = torch.tensor(X_stat_test.values, dtype=torch.float32).to(device)  # THÊM .values  # SỬA Ở ĐÂY

# Load model (cần class model đã định nghĩa)
# Tạo model với cấu hình GIỐNG HỆT lúc train
# Lấy num_classes từ encoders
num_classes = [len(encoders[col].classes_) for col in encoders.keys()]
print("num_classes:", num_classes)

# Tạo model
model = ImprovedBiLSTM_GCN_Model(
    num_actions=len(processor.action2idx) + 1,
    num_classes=num_classes,  # Giờ đã có
    dropout=0.15,
    use_attention=False
).to(device)


model.load_state_dict(torch.load(drive_path + "best_model.pth"))
model.eval()
print("✅ Loaded model!")

# Predict
with torch.no_grad():
    outputs = model(X_seq_test, X_mask_test, X_stat_test)

print(f"Số outputs: {len(outputs)}")
for i, out in enumerate(outputs):
    print(f"Output {i} shape: {out.shape}")

num_classes: [12, 31, 99, 12, 31, 99]
✅ Loaded model!
Số outputs: 6
Output 0 shape: torch.Size([38000, 12])
Output 1 shape: torch.Size([38000, 31])
Output 2 shape: torch.Size([38000, 99])
Output 3 shape: torch.Size([38000, 12])
Output 4 shape: torch.Size([38000, 31])
Output 5 shape: torch.Size([38000, 99])


In [ ]:
# Lấy predictions (argmax)
preds = []
for i, out in enumerate(outputs):
    pred = torch.argmax(out, dim=1).cpu().numpy()
    preds.append(pred)
    print(f"Head {i}: min={pred.min()}, max={pred.max()}")

preds = np.stack(preds, axis=1)
print(f"Predictions shape: {preds.shape}")

Head 0: min=0, max=11
Head 1: min=0, max=30
Head 2: min=0, max=98
Head 3: min=0, max=11
Head 4: min=0, max=30
Head 5: min=0, max=98
Predictions shape: (38000, 6)


In [ ]:
# Decode predictions
decoded_preds = []

for i, col in enumerate(encoders.keys()):
    decoded = encoders[col].inverse_transform(preds[:, i])
    decoded_preds.append(decoded)
    print(f"{col}: {decoded[:5]}")  # In 5 mẫu đầu

decoded_preds = np.stack(decoded_preds, axis=1)
print(f"Decoded shape: {decoded_preds.shape}")

attr_1: [11  9  1  5 11]
attr_2: [ 1 19  1  1  1]
attr_3: [84 55 98 72 22]
attr_4: [12  3  3  1 12]
attr_5: [ 1 12  1  1  1]
attr_6: [96 63 53 77  1]
Decoded shape: (38000, 6)


In [ ]:
# Tạo DataFrame từ decoded predictions
submission = pd.DataFrame(decoded_preds, columns=encoders.keys())

# Thêm cột id
submission.insert(0, "id", X_test["id"].values)

# Chỉ ép kiểu các cột attr (6 cột)
for col in encoders.keys():
    submission[col] = submission[col].astype(np.uint16)

# Lưu file
output_file = "ten_doi_thi.csv"
submission.to_csv(output_file, index=False)

print(f"✅ Đã tạo file: {output_file}")
print(f"Kích thước: {os.path.getsize(output_file)/1024:.2f} KB")

# Hiển thị 5 dòng đầu
print("\n📊 5 dòng đầu tiên:")
print(submission.head())

# Tải về máy
from google.colab import files
files.download(output_file)
print(f"\n✅ Đã tải {output_file} về máy!")

✅ Đã tạo file: ten_doi_thi.csv
Kích thước: 780.63 KB

📊 5 dòng đầu tiên:
      id  attr_1  attr_2  attr_3  attr_4  attr_5  attr_6
0  gpbfd      11       1      84      12       1      96
1  w22ee       9      19      55       3      12      63
2  wyw95       1       1      98       3       1      53
3  izx4w       5       1      72       1       1      77
4  c6o2d      11       1      22      12       1       1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Đã tải ten_doi_thi.csv về máy!


In [ ]:
import pandas as pd
import numpy as np

# Đọc 2 file
df_ground_truth = pd.read_csv('submission.csv')  # File cũ (coi là kết quả đúng)
df_new = pd.read_csv('ten_doi_thi.csv')  # File mới của cậu

print("="*60)
print("🎯 SO SÁNH ACCURACY VỚI FILE KẾT QUẢ")
print("="*60)

# 1. Kiểm tra shape
print(f"\n1️⃣ KÍCH THƯỚC:")
print(f"   File kết quả: {df_ground_truth.shape}")
print(f"   File mới: {df_new.shape}")

if df_ground_truth.shape != df_new.shape:
    print("❌ 2 file khác kích thước! Không thể so sánh trực tiếp.")
    # Lấy min shape để so sánh
    min_rows = min(len(df_ground_truth), len(df_new))
    df_ground_truth = df_ground_truth.iloc[:min_rows]
    df_new = df_new.iloc[:min_rows]
    print(f"   → So sánh trên {min_rows} dòng đầu")

# 2. Sắp xếp theo id
df_ground_truth = df_ground_truth.sort_values('id').reset_index(drop=True)
df_new = df_new.sort_values('id').reset_index(drop=True)

# 3. TÍNH EXACT MATCH ACCURACY (QUAN TRỌNG NHẤT)
attr_cols = ['attr_1', 'attr_2', 'attr_3', 'attr_4', 'attr_5', 'attr_6']
total = len(df_ground_truth)
correct = 0
wrong_samples = []

print(f"\n2️⃣ EXACT MATCH ACCURACY (đúng cả 6 attr):")

for i in range(total):
    ground_vals = df_ground_truth.loc[i, attr_cols].values
    new_vals = df_new.loc[i, attr_cols].values

    if np.array_equal(ground_vals, new_vals):
        correct += 1
    else:
        wrong_samples.append({
            'index': i,
            'id': df_ground_truth.loc[i, 'id'],
            'ground_truth': ground_vals,
            'new': new_vals,
            'diff': [j+1 for j in range(6) if ground_vals[j] != new_vals[j]]
        })

accuracy = correct / total * 100
print(f"   ✅ Đúng hoàn toàn: {correct}/{total} = {accuracy:.2f}%")
print(f"   ❌ Sai: {total - correct}/{total} = {100-accuracy:.2f}%")

# 4. ACCURACY TỪNG ATTR RIÊNG LẺ
print(f"\n3️⃣ ACCURACY TỪNG ATTR (không cần đúng cả 6):")

for col in attr_cols:
    correct_attr = (df_ground_truth[col] == df_new[col]).sum()
    acc_attr = correct_attr / total * 100
    print(f"   {col}: {correct_attr}/{total} = {acc_attr:.2f}%")

# 5. PHÂN TÍCH MẪU SAI
print(f"\n4️⃣ PHÂN TÍCH {len(wrong_samples)} MẪU SAI:")

if wrong_samples:
    # Đếm số attr sai trong mỗi mẫu
    sai_counts = {1:0, 2:0, 3:0, 4:0, 5:0, 6:0}

    for sample in wrong_samples:
        so_attr_sai = len(sample['diff'])
        sai_counts[so_attr_sai] = sai_counts.get(so_attr_sai, 0) + 1

    print(f"\n   📊 Phân bố số attr sai trong 1 mẫu:")
    for k in range(1, 7):
        if k in sai_counts and sai_counts[k] > 0:
            print(f"      Sai {k} attr: {sai_counts[k]} mẫu ({sai_counts[k]/len(wrong_samples)*100:.1f}%)")

    # Hiển thị 10 mẫu sai đầu tiên
    print(f"\n   📝 10 mẫu sai đầu tiên:")
    for i, sample in enumerate(wrong_samples[:10]):
        print(f"\n   Mẫu {i+1} - ID: {sample['id']}")
        print(f"      Kết quả: {sample['ground_truth']}")
        print(f"      Dự đoán: {sample['new']}")
        print(f"      Sai attr: {sample['diff']}")

# 6. THỐNG KÊ CƠ BẢN
print(f"\n5️⃣ THỐNG KÊ CƠ BẢN:")

for col in attr_cols:
    print(f"\n   {col}:")
    print(f"     Kết quả: min={df_ground_truth[col].min()}, max={df_ground_truth[col].max()}, mean={df_ground_truth[col].mean():.2f}")
    print(f"     Dự đoán: min={df_new[col].min()}, max={df_new[col].max()}, mean={df_new[col].mean():.2f}")

# 7. MA TRẬN TƯƠNG QUAN
print(f"\n6️⃣ MA TRẬN TƯƠNG QUAN GIỮA CÁC ATTR (dự đoán):")
corr_matrix = df_new[attr_cols].corr()
print(corr_matrix.round(3))

# 8. TỔNG KẾT
print("\n" + "="*60)
print("🎯 TỔNG KẾT:")
print("="*60)
print(f"📊 EXACT MATCH ACCURACY: {accuracy:.2f}%")
print(f"   (So với file kết quả)")

if accuracy > 95:
    print("🎉 GẦN NHƯ GIỐNG HOÀN TOÀN!")
elif accuracy > 90:
    print("👍 RẤT TỐT! Chỉ khác một chút!")
elif accuracy > 80:
    print("📈 KHÁ! Còn sai sót nhưng ổn!")
elif accuracy > 70:
    print("🔄 TRUNG BÌNH! Cần xem lại model!")
else:
    print("🔧 YẾU! Model đang dự đoán khác nhiều!")

🎯 SO SÁNH ACCURACY VỚI FILE KẾT QUẢ

1️⃣ KÍCH THƯỚC:
   File kết quả: (38000, 7)
   File mới: (38000, 7)

2️⃣ EXACT MATCH ACCURACY (đúng cả 6 attr):
   ✅ Đúng hoàn toàn: 32964/38000 = 86.75%
   ❌ Sai: 5036/38000 = 13.25%

3️⃣ ACCURACY TỪNG ATTR (không cần đúng cả 6):
   attr_1: 35648/38000 = 93.81%
   attr_2: 37582/38000 = 98.90%
   attr_3: 37526/38000 = 98.75%
   attr_4: 36770/38000 = 96.76%
   attr_5: 37165/38000 = 97.80%
   attr_6: 37071/38000 = 97.56%

4️⃣ PHÂN TÍCH 5036 MẪU SAI:

   📊 Phân bố số attr sai trong 1 mẫu:
      Sai 1 attr: 4002 mẫu (79.5%)
      Sai 2 attr: 877 mẫu (17.4%)
      Sai 3 attr: 146 mẫu (2.9%)
      Sai 4 attr: 11 mẫu (0.2%)

   📝 10 mẫu sai đầu tiên:

   Mẫu 1 - ID: a03zc
      Kết quả: [np.int64(4) np.int64(16) np.int64(60) np.int64(2) np.int64(2)
 np.int64(52)]
      Dự đoán: [np.int64(4) np.int64(16) np.int64(60) np.int64(6) np.int64(2)
 np.int64(52)]
      Sai attr: [4]

   Mẫu 2 - ID: a04bw
      Kết quả: [np.int64(2) np.int64(12) np.int64(25) np.int6